In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import statsmodels.api as sm
import seaborn as sns
import os
from statsmodels.nonparametric.smoothers_lowess import lowess
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from scipy.stats import boxcox 
from statsmodels.tsa.seasonal import STL
from utils import load_data, check_stationarity

# Load & Process Data

In [ ]:
selected_data = load_data()

## Plot time Series

In [ ]:
def plot_time_series(df, date_col, var_col, ma_window=None):
    # 1. Set up the plot
    fig, ax = plt.subplots(figsize=(12, 6))

    # 2. Plot using the new 'date' column for the x-axis
    ax.plot(df[date_col], df[var_col], marker='o', linestyle='-')
    # 3. Optionally plot moving average
    if ma_window is not None and ma_window > 1:
        ma_series = df[var_col].rolling(window=ma_window).mean()
        ax.plot(df[date_col], ma_series, color='red', linewidth=2,
                label=f'{ma_window}-period MA')
        
    # 3. Format the date axis for clarity ✨
    # Set the major locator to find the start of each year
    ax.xaxis.set_major_locator(mdates.YearLocator(base=5))
    # Set the format of the major labels to show just the year (e.g., "2023")
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

    # To add ticks for every 3 months, you can use a minor locator
    ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=6))

    # 4. Add labels and a grid
    ax.set_title(var_col)
    ax.set_xlabel('Date')
    ax.set_ylabel('Value')
    ax.grid(True, which='major', alpha=0.6)
    ax.grid(True, which='minor', alpha=0.2)

    plt.tight_layout()
    plt.show()


In [ ]:
plot_time_series(selected_data, 'date', 'tdiff', ma_window=12)

In [ ]:
plot_time_series(selected_data, 'date', 'tmed', ma_window=12)

In [ ]:
plot_time_series(selected_data, 'date', 'tmax', ma_window=12)

In [ ]:
plot_time_series(selected_data, 'date', 'tmin', ma_window=12)

In [ ]:
plot_time_series(selected_data, 'date', 'prec', ma_window=12)

### Log Feat

In [ ]:
plot_time_series(selected_data, 'date', 'tdiff', ma_window=12)

## Seasonal Plot

In [ ]:
temp_data = selected_data[['date','year','month', 'tdiff', 'prec', 'tmax', 'tmin', 'log_diff']]
temp_data = temp_data.set_index('date')

In [ ]:
def seasonal_plot(df, col, title):
    _, ax = plt.subplots(figsize=(16, 8))
    sm.graphics.tsa.month_plot(df[col], ylabel='col', ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Month")
    plt.show()

In [ ]:
seasonal_plot(temp_data, 'tdiff', "Seasonal Subseries Plot: Temperature Range (Max - Min)")

In [ ]:
seasonal_plot(temp_data, 'log_diff', "Seasonal Subseries Plot: Log Temperature Range (Max - Min)")

## Box Cox

In [ ]:
bc, lambda_ = boxcox(temp_data['tdiff'])
print(f"Estimated Lambda: {lambda_}")

plot_time_series(pd.DataFrame.from_dict({
    "date": list(temp_data.index),
    "box_cox": bc
}), 'date', 'box_cox', ma_window=12)

In [ ]:
bc, lambda_ = boxcox(temp_data['log_diff'])
print(f"Estimated Lambda: {lambda_}")

plot_time_series(pd.DataFrame.from_dict({
    "date": list(temp_data.index),
    "box_cox": bc
}), 'date', 'box_cox', ma_window=12)

## Linear Regression

In [ ]:
import statsmodels.formula.api as smf 
temp_data['date'] = temp_data.index
temp_data['time'] = temp_data['date'].dt.to_period('M').astype(int) 

# fir linear regression model
fit = smf.ols(formula='tdiff ~ time', data=temp_data).fit()
print(fit.summary())

In [ ]:
residuals = fit.resid
# Resíduos: detrended time series
plt.subplot(1, 1, 1)
plt.plot(temp_data['date'], residuals, color='blue')
plt.title("Temperature Range Detrended")
plt.ylabel("Residuals")
plt.xlabel("Time")

plt.show()

In [ ]:
import statsmodels.formula.api as smf 

temp_data['time'] = temp_data['date'].dt.to_period('M').astype(int) 

# fir linear regression model
fit = smf.ols(formula='tdiff ~ time', data=temp_data).fit()
print(fit.summary())

residuals = fit.resid
# Resíduos: detrended time series
plt.subplot(1, 1, 1)
plt.plot(temp_data['date'], residuals, color='blue')
plt.title("Temperature Range Detrended")
plt.ylabel("Residuals")
plt.xlabel("Time")

plt.show()

## Difference Operator

In [ ]:
tdiff_series = temp_data['tdiff']
tdiff_diff = tdiff_series.diff(1)
tdiff_diff

In [ ]:
plt.plot(temp_data.index, tdiff_diff, color='#ff9900')
plt.title("Temperature Range Differenced")
plt.ylabel("Temperature Range")
plt.xlabel("date")

In [ ]:
diff12 = tdiff_diff.to_frame()
diff12['year_month'] = pd.to_datetime(diff12.index)
diff12

In [ ]:
diff12 = tdiff_diff.to_frame()
diff12['year_month'] = pd.to_datetime(diff12.index)

x = diff12[['year_month','tdiff']]
x = x.set_index('year_month')

seasonal_plot(x, 'tdiff', "Seasonal Plot: tdiff diff 12")


## STL Decomposition

In [ ]:
# STL with periodic seasonal component
stl_1 = STL(temp_data['log_diff'], seasonal=len(temp_data), robust=True).fit()

# STL with seasonal window of 13
stl_2 = STL(temp_data['log_diff'], seasonal=13, robust=True).fit()

In [ ]:
print("Periodic Season")
stl_1.plot()

In [ ]:
print("13 Season")
stl_2.plot()

In [ ]:
# STL with periodic seasonal component
stl_1 = STL(temp_data['tdiff'], seasonal=len(temp_data), robust=False).fit()

# STL with seasonal window of 13
stl_2 = STL(temp_data['tdiff'], seasonal=13, robust=False).fit()

stl_1.plot()
stl_2.plot()

In [ ]:

from statsmodels.stats.diagnostic import het_arch
print(het_arch(stl_1.resid)) # Not heteroskedascity

In [ ]:
print(het_arch(stl_1.resid, nlags=12)) 

## Lag Plots

In [ ]:
def lag_plot_grid(ts, ys=None, lags=12, title="Lag Plots"):
    """Create grid of lag plots"""
    fig, axes = plt.subplots(3, 4, figsize=(15, 10))
    fig.suptitle(title, fontsize=16)
    
    for i in range(lags):
        row = i // 4
        col = i % 4
        
        # Create lagged series
        if ys is not None:
            lagged = ys.shift(i+1)
        else:
            lagged = ts.shift(i+1)
        
        # Remove NaN values
        mask = ~(np.isnan(ts) | np.isnan(lagged))
        x = ts[mask]
        y = lagged[mask]
        
        # Scatter plot
        axes[row, col].scatter(y, x, alpha=0.9, s=10, color="steelblue", edgecolor="black")
        axes[row, col].set_title(f'Lag {i+1}')
        axes[row, col].set_ylabel('X(t)')
        if ys is not None:
            axes[row, col].set_xlabel(f'Y(t-{i+1})')
        else:
            axes[row, col].set_xlabel(f'X(t-{i+1})')
        axes[row, col].grid(True, alpha=0.3)

        # Compute correlation
        corr = np.corrcoef(y, x)[0, 1]
        axes[row, col].text(
            0.05, 0.95,
            f"r = {corr:.3f}",
            transform=axes[row, col].transAxes,
            fontsize=12,
            verticalalignment="top",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.5)
        )

        # Fit LOWESS
        smoothed = lowess(x, y, frac=0.3)  # frac controls smoothing
        axes[row, col].plot(smoothed[:,0], smoothed[:,1], color="red", linewidth=1.5)

    plt.tight_layout()
    plt.show()


In [ ]:
lag_plot_grid(temp_data['tdiff'], ys=temp_data['prec'], title="Lag Plots: Tdiff v Prec")

In [ ]:
lag_plot_grid(temp_data['tdiff'], ys=temp_data['tmax'], title="Lag Plots: Tdiff v Tmax")

In [ ]:
lag_plot_grid(temp_data['tdiff'], ys=temp_data['tmin'], title="Lag Plots: Tdiff v Tmin")

In [ ]:
lag_plot_grid(temp_data['tdiff'], title="Lag Plots: Tdiff")

In [ ]:
lag_plot_grid(temp_data['log_diff'], title="Lag Plots: Log Tdiff")
lag_plot_grid(stl_1.resid, title="Lag Plots: Log Tdiff - Remainder")

## ACF

In [ ]:
plot_acf(temp_data['prec'])
plt.show()

In [ ]:
plot_acf(temp_data['tdiff'])
plt.show()

In [ ]:
plot_pacf(temp_data['tdiff'])
plt.show()

In [ ]:
temp_data['date'] = temp_data.index
plot_time_series(temp_data, 'date', 'tdiff', ma_window=12)

## Stationarity

In [ ]:
check_stationarity(temp_data['tdiff'])

In [ ]:
check_stationarity(temp_data['tdiff'].diff().dropna())

In [ ]:
tdiff_diff = temp_data['tdiff']
plot_acf(tdiff_diff)
plot_pacf(tdiff_diff)
plt.show()